# Jevlet on Colab: public data, resumable training, tiny ablations

This notebook has two deliberately separate tracks. The **daily-driver** uses a small pretrained semantic encoder on the laptop; the **research track** trains Jevlet-S from scratch and tests architecture hypotheses. A scratch checkpoint is not promoted to daily use just because it trains.

Success criteria: pinned public data with disjoint train/dev/vault splits; a resumable checkpoint copied off the ephemeral VM; and a fixed-budget pointer-vs-bilinear comparison. The vault is generated but never passed to the research runner. For strict vault blindness, evaluate it in a separate account after freezing the architecture.

Set a GPU runtime in Colab. An L4 is preferable if offered; T4 is usable. Colab hardware is not guaranteed. This notebook inspects the assigned GPU and **does not install CUDA drivers**.

In [ ]:
from __future__ import annotations

import json
import shutil
import subprocess
import sys
from pathlib import Path

import torch

REPO_URL = ""  # Set after pushing your repository, e.g. https://github.com/you/Jevlet.git
DRIVE_SOURCE_FOLDER = "/content/drive/MyDrive/Jevlet-source"  # alternative to REPO_URL
PERSIST_TO = "drive"  # "drive" or "hf"
HF_REPO_ID = ""  # required only for HF; use a private model repo
PUBLIC_DATASETS = ["banking77", "boolq", "mnli"]
ALLOW_UNCLEAR_DATASET_LICENSES = False  # opt in before using AG News, SST-2, Yelp
SEED = 1337
PUBLIC_STEPS = 150
ABLATION_STEPS = 60
RUN_ROUTER_SMOKE = True
RUN_ABLATIONS = True

print("torch", torch.__version__, "torch CUDA build", torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Runtime > Change runtime type before training")
print("GPU:", torch.cuda.get_device_name(0), "capability:", torch.cuda.get_device_capability(0))
print("CUDA visible:", torch.cuda.is_available(), "BF16 supported:", torch.cuda.is_bf16_supported())
subprocess.run(["nvidia-smi"], check=True)

## Source and dependencies

Point to your pushed Git repository or copy the source folder to Drive. The working copy stays on `/content` for speed. Drive/HF keeps checkpoints, not a live training filesystem. Authentication happens through Colab/Hugging Face prompts; no token is stored in this notebook.

In [ ]:
from google.colab import drive

if PERSIST_TO == "drive" or not REPO_URL:
    drive.mount("/content/drive")
PROJECT = Path("/content/Jevlet")
if REPO_URL:
    if not PROJECT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT)], check=True)
else:
    source = Path(DRIVE_SOURCE_FOLDER)
    if not source.is_dir():
        raise FileNotFoundError(
            "Set REPO_URL after pushing, or place Jevlet source in DRIVE_SOURCE_FOLDER"
        )
    if not PROJECT.exists():
        shutil.copytree(source, PROJECT)
if not (PROJECT / "pyproject.toml").exists():
    raise RuntimeError("Selected source is not a Jevlet repository")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        str(PROJECT),
        "datasets",
        "huggingface_hub",
        "sentence-transformers",
    ],
    check=True,
)
sys.path.insert(0, str(PROJECT))
from jevlet.public_data import download_public_data  # noqa: E402
from jevlet.synthetic import generate_dataset  # noqa: E402
from jevlet.training import train_experiment  # noqa: E402
from notebooks.colab_runtime import CheckpointSync, amp_dtype, restore_checkpoint  # noqa: E402

print(
    "source:",
    PROJECT,
    "commit:",
    subprocess.check_output(["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True).strip()
    if (PROJECT / ".git").exists()
    else "Drive copy",
)
print("mixed precision:", amp_dtype(torch))

## Persistence and dataset provenance

The first download records each source's exact Hugging Face revision, license card, split counts, and SHA-256. On resume the same revisions and output hashes are required. AG News/SST-2 have unknown card licenses and Yelp has special terms; add them only after reviewing terms and setting the opt-in above. BANKING77 uses the parquet-backed MTEB mirror because the original repository's obsolete loader script is unsupported in current Datasets.

In [ ]:
RUNS = Path("/content/jevlet_runs")
RUNS.mkdir(parents=True, exist_ok=True)
drive_checkpoint_root = (
    Path("/content/drive/MyDrive/Jevlet/checkpoints") if PERSIST_TO == "drive" else None
)
if PERSIST_TO == "hf":
    if not HF_REPO_ID:
        raise ValueError("Set HF_REPO_ID to a private model repository")
    from huggingface_hub import HfApi, notebook_login

    notebook_login()
    HfApi().create_repo(repo_id=HF_REPO_ID, repo_type="model", private=True, exist_ok=True)
elif PERSIST_TO != "drive":
    raise ValueError("PERSIST_TO must be drive or hf")
stored = restore_checkpoint(
    "dataset_manifest.json", RUNS, drive_root=drive_checkpoint_root, hf_repo_id=HF_REPO_ID or None
)
old_manifest = json.loads(stored.read_text()) if stored else None
locked_revisions = (
    {name: source["revision"] for name, source in old_manifest["sources"].items()}
    if old_manifest
    else None
)
PUBLIC_DIR = Path("/content/jevlet_data/public")
manifest = download_public_data(
    PUBLIC_DIR,
    PUBLIC_DATASETS,
    seed=SEED,
    train_limit=1200,
    dev_limit=200,
    vault_limit=200,
    allow_unknown_license=ALLOW_UNCLEAR_DATASET_LICENSES,
    revisions=locked_revisions,
)
if old_manifest:
    for split in ("train", "dev", "vault"):
        if manifest["splits"][split]["sha256"] != old_manifest["splits"][split]["sha256"]:
            raise RuntimeError(f"{split} dataset hash changed; do not resume this checkpoint")
shutil.copy2(PUBLIC_DIR / "manifest.json", RUNS / "dataset_manifest.json")
CheckpointSync(RUNS, drive_root=drive_checkpoint_root, hf_repo_id=HF_REPO_ID or None).sync_once()
print(
    json.dumps(
        {
            "sources": manifest["sources"],
            "split_counts": {
                split: manifest["splits"][split]["rows"] for split in ("train", "dev", "vault")
            },
        },
        indent=2,
    )
)

## Daily-driver compatibility smoke

This is not training: it confirms the pretrained local router can load and score dynamic choices. It should remain confirmation-gated until feedback calibration is validated; Colab is not the always-on desktop host.

In [ ]:
if RUN_ROUTER_SMOKE:
    from jevlet.semantic import Candidate, SemanticRouter

    router = SemanticRouter()
    sample = router.decision(
        "Write and test a Python bug fix",
        "Who should handle this task?",
        [
            Candidate("Codex", "Write and test code"),
            Candidate("Human", "Ask a person for judgment"),
        ],
    )
    print(
        {
            "selected": sample.choice.selected,
            "probabilities": sample.choice.probabilities,
            "gate": sample.gate,
            "calibrated": sample.calibrated,
        }
    )

## Public-data scratch baseline

This run proves the public adapter, model, evaluation, and full-state checkpoint path work together. It is **not** expected to beat the pretrained daily-driver after only 150 scratch steps. Increase `PUBLIC_STEPS` on rerun to continue the same checkpoint. The local checkpoint syncs periodically and again at completion.

In [ ]:
PUBLIC_RUN = RUNS / "public_baseline"
public_config = json.loads((PROJECT / "configs/proxy.json").read_text())
public_config["seed"] = SEED
public_config["model"]["max_seq_len"] = 512
public_config["data"].update(
    train=str(PUBLIC_DIR / "train.jsonl"),
    dev=str(PUBLIC_DIR / "dev.jsonl"),
    max_state_bytes=160,
    max_question_bytes=80,
    max_option_bytes=48,
)
public_config["training"].update(
    max_steps=PUBLIC_STEPS,
    batch_size=2,
    gradient_accumulation=8,
    amp_dtype=amp_dtype(torch),
    save_every_steps=25,
)
public_config["evaluation"].update(batch_size=4, permutation_examples=8, benchmark_repeats=2)
resumed = restore_checkpoint(
    "public_baseline/last.pt", RUNS, drive_root=drive_checkpoint_root, hf_repo_id=HF_REPO_ID or None
)
if resumed:
    public_config["training"]["resume_from"] = str(resumed)
    print("resuming from", resumed)
sync = CheckpointSync(
    RUNS, drive_root=drive_checkpoint_root, hf_repo_id=HF_REPO_ID or None, interval_seconds=180
)
sync.start()
try:
    public_metrics = train_experiment(public_config, PUBLIC_RUN)
finally:
    synced = sync.stop()
print(
    {
        key: public_metrics.get(key)
        for key in (
            "accuracy",
            "ood_accuracy",
            "ece",
            "brier",
            "peak_vram_mb",
            "steps",
            "checkpoint",
        )
    }
)
print("off-VM files synced:", len(synced), "total tracked:", len(sync.synced))

## Fixed-budget architecture ablations

The following pair changes only the decision head (pointer vs bilinear) on the same synthetic split, seed, and step budget. It records accuracy, OOD accuracy, Brier, ECE, unknown accuracy, throughput, and VRAM. These are preliminary comparisons, not proof of an architecture winner. The hidden vault is not opened here.

In [ ]:
synthetic_dir = Path("/content/jevlet_data/synthetic")
synthetic_manifest = generate_dataset(synthetic_dir, count=5000, seed=SEED)
print({name: item["count"] for name, item in synthetic_manifest["splits"].items()})
ablation_results = []
if RUN_ABLATIONS:
    for head in ("pointer", "bilinear"):
        config = json.loads((PROJECT / "configs/proxy.json").read_text())
        config["seed"] = SEED
        config["model"]["decision_head"] = head
        config["data"].update(
            train=str(synthetic_dir / "train.jsonl"), dev=str(synthetic_dir / "dev.jsonl")
        )
        config["training"].update(
            max_steps=ABLATION_STEPS,
            batch_size=2,
            gradient_accumulation=8,
            amp_dtype=amp_dtype(torch),
            save_every_steps=20,
        )
        config["evaluation"].update(batch_size=4, permutation_examples=8, benchmark_repeats=2)
        run_dir = RUNS / f"ablation_{head}"
        resumed = restore_checkpoint(
            f"ablation_{head}/last.pt",
            RUNS,
            drive_root=drive_checkpoint_root,
            hf_repo_id=HF_REPO_ID or None,
        )
        if resumed:
            config["training"]["resume_from"] = str(resumed)
        sync = CheckpointSync(
            RUNS,
            drive_root=drive_checkpoint_root,
            hf_repo_id=HF_REPO_ID or None,
            interval_seconds=180,
        )
        sync.start()
        try:
            metrics = train_experiment(config, run_dir)
        finally:
            sync.stop()
        ablation_results.append(
            {
                "head": head,
                **{
                    key: metrics.get(key)
                    for key in (
                        "accuracy",
                        "ood_accuracy",
                        "brier",
                        "ece",
                        "unknown_accuracy",
                        "train_questions_per_second",
                        "peak_vram_mb",
                        "steps",
                    )
                },
            }
        )
print(json.dumps(ablation_results, indent=2))

## Interpretation and next run

Compare the measured trade-offs, not just one score. Keep the pretrained router as the day-one service unless a scratch model wins on held-out user-relevant cases, calibration, and latency. For a longer successive-halving night, use `scripts/run_overnight.py` with a frozen synthetic manifest and an explicit hours budget; no vault labels should enter that search. Before claiming generalization, freeze the candidate and evaluate the vault separately.

Colab can disconnect despite a paid plan. Verify that `last.pt` exists in Drive or the private HF repo before closing the session; rerun this notebook with the same settings and a larger step target to resume. Public data and model downloads require network access and may change terms; the manifest records the source revisions.